# One-Shot Document Digitizer — Google Colab

Run Baidu's Unlimited-OCR model on a free T4 GPU.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom
3. Upload your PDF/images when prompted
4. Get structured OCR output

In [ ]:
#@title 1. Install Dependencies (run once)
!pip install -q pymupdf transformers==4.57.1 einops addict easydict Pillow matplotlib psutil
!pip install -q flask pyngrok > /dev/null 2>&1
print('Done')

In [ ]:
#@title 2. Load Model (takes ~30s)
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = 'baidu/Unlimited-OCR'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print('Loading model...')
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
)
model = model.cuda().eval()

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f'Model loaded on {gpu_name} ({gpu_mem:.1f} GB)')

In [ ]:
#@title 3. OCR Functions
import fitz  # PyMuPDF
from pathlib import Path
import shutil
import json

OUTPUT_DIR = Path('/content/ocr_output')
OUTPUT_DIR.mkdir(exist_ok=True)

def pdf_to_images(pdf_path, dpi=200):
    """Convert PDF pages to images."""
    doc = fitz.open(pdf_path)
    page_dir = OUTPUT_DIR / 'pages'
    page_dir.mkdir(exist_ok=True)
    images = []
    zoom = dpi / 72
    matrix = fitz.Matrix(zoom, zoom)
    for i in range(len(doc)):
        pix = doc[i].get_pixmap(matrix=matrix)
        path = str(page_dir / f'page_{i+1:04d}.png')
        pix.save(path)
        images.append(path)
    doc.close()
    print(f'Converted {len(images)} pages to images')
    return images

def ocr_single(image_path):
    """OCR a single image."""
    out = str(OUTPUT_DIR / 'single')
    model.infer(
        tokenizer,
        prompt='<image>document parsing.',
        image_file=image_path,
        output_path=out,
        base_size=1024,
        image_size=640,
        crop_mode=True,
        max_length=32768,
        no_repeat_ngram_size=35,
        ngram_window=128,
        save_results=True,
    )
    return read_outputs(out)

def ocr_multi(image_paths):
    """OCR multiple pages in one pass."""
    out = str(OUTPUT_DIR / 'multi')
    model.infer_multi(
        tokenizer,
        prompt='<image>Multi page parsing.',
        image_files=image_paths,
        output_path=out,
        image_size=1024,
        max_length=32768,
    )
    return read_outputs(out)

def read_outputs(output_dir):
    result = {'text': None, 'markdown': None, 'json': None}
    p = Path(output_dir)
    if not p.exists():
        return result
    for f in p.iterdir():
        if f.suffix == '.txt':
            result['text'] = f.read_text(encoding='utf-8')
        elif f.suffix == '.md':
            result['markdown'] = f.read_text(encoding='utf-8')
        elif f.suffix == '.json':
            result['json'] = f.read_text(encoding='utf-8')
    base = result['text'] or result['markdown'] or ''
    if not result['text']: result['text'] = base
    if not result['markdown']: result['markdown'] = base
    if not result['json']: result['json'] = base
    return result

print('OCR functions ready')

In [ ]:
#@title 4. Upload & OCR
import google.colab.files as files

print('Upload a PDF or image:')
uploaded = files.upload()

if not uploaded:
    print('No file uploaded')
else:
    filename = list(uploaded.keys())[0]
    ext = Path(filename).suffix.lower()
    print(f'\nProcessing: {filename}')

    if ext == '.pdf':
        images = pdf_to_images(filename)
        print(f'Running multi-page OCR on {len(images)} pages...')
        result = ocr_multi(images)
    else:
        print('Running single-image OCR...')
        result = ocr_single(filename)

    print('\n' + '='*60)
    print('PLAIN TEXT OUTPUT')
    print('='*60)
    print(result['text'] or '(no output)')

In [ ]:
#@title 5. Download Results
from google.colab import files as colab_files

if 'result' in dir() and result:
    for fmt in ['text', 'markdown']:
        if result.get(fmt):
            ext = 'md' if fmt == 'markdown' else 'txt'
            out_path = f'/content/ocr_result.{ext}'
            with open(out_path, 'w', encoding='utf-8') as f:
                f.write(result[fmt])
            print(f'Downloading ocr_result.{ext}')
            colab_files.download(out_path)
    print('\nDone!')
else:
    print('No results to download yet — run cell 4 first.')